In [2]:
# Celda E1 - Configuracion y carga de la matriz global
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path('/Users/ppizam/Claude/Master Thesis')
MATRIX = BASE / 'Desarrollo' / 'Metodologia' / 'Matrix'
(MATRIX / 'eventos').mkdir(exist_ok=True)

# parametros del detector v1 (calibrables)
VENTANA = 30      # dias de linea base movil
MIN_P = 15        # minimo de dias para tener base valida
Z_ON = 4.0        # umbral de encendido (desviaciones sobre la base)
MIN_ABS = 30      # menciones absolutas minimas el dia de encendido
S_OFF = 1.0       # umbral de extincion: mu0 + S_OFF * sigma0
K_OFF = 3         # dias consecutivos bajo el umbral para declarar fin
GAP = 5           # fusionar eventos del mismo ticker separados por <= GAP dias

g = pd.read_csv(MATRIX / 'maestras' / 'matriz_global_2020_2026.csv',
                index_col=0, keep_default_na=False, na_values=[''])
g.index = pd.to_datetime(g.index)
print('matriz global:', g.shape)

# universo operativo: tickers que alguna vez tuvieron un dia con >= MIN_ABS menciones
candidatos = g.columns[(g.max(axis=0) >= MIN_ABS)]
gc = g[candidatos]
print(f'tickers candidatos a evento (max diario >= {MIN_ABS}): {len(candidatos)}')

# lineas base moviles (excluyen el dia actual via shift)
mu = gc.rolling(VENTANA, min_periods=MIN_P).mean().shift(1)
sd = gc.rolling(VENTANA, min_periods=MIN_P).std().shift(1)
sd_piso = np.maximum(np.maximum(sd, np.sqrt(mu)), 1.0)
z = (gc - mu) / sd_piso
print('lineas base listas')

matriz global: (2373, 8812)
tickers candidatos a evento (max diario >= 30): 2658
lineas base listas


In [3]:
# Celda E2 v2 - Deteccion con extincion relativa al pico + catalogo en dos capas
KAPPA = 0.25    # umbral de extincion: KAPPA * pico corrido del evento
K_OFF = 5      # dias consecutivos bajo el umbral para declarar fin
MIN_DUR = 3    # filtros del catalogo principal
MIN_MENC = 300

FECHA_FIN_VENTANA = gc.index[-1]

eventos = []
for t in gc.columns:
    m = gc[t].to_numpy()
    zz = z[t].to_numpy()
    mus = mu[t].to_numpy()
    sds = sd_piso[t].to_numpy()
    fechas = gc.index

    crudos = []
    en_evento = False
    for i in range(len(m)):
        if not en_evento:
            if not np.isnan(zz[i]) and zz[i] >= Z_ON and m[i] >= MIN_ABS:
                en_evento = True
                i0 = i
                mu0, s0 = mus[i], sds[i]
                z0 = zz[i]
                pico_corrido = m[i]
                bajo = 0
        else:
            if m[i] > pico_corrido:
                pico_corrido = m[i]
            umbral_fin = max(mu0 + S_OFF * s0, KAPPA * pico_corrido)
            if m[i] < umbral_fin:
                bajo += 1
                if bajo >= K_OFF:
                    crudos.append((i0, i - K_OFF, mu0, s0, z0, False))
                    en_evento = False
            else:
                bajo = 0
    if en_evento:
        crudos.append((i0, len(m) - 1, mu0, s0, z0, True))   # censurado

    # fusionar eventos del mismo ticker separados por <= GAP dias
    fusion = []
    for ev in crudos:
        if fusion and ev[0] - fusion[-1][1] <= GAP:
            prev = fusion[-1]
            fusion[-1] = (prev[0], ev[1], prev[2], prev[3], prev[4], ev[5])
        else:
            fusion.append(ev)

    for i0, i1, mu0, s0, z0, cens in fusion:
        tramo = m[i0:i1 + 1]
        ipico = i0 + int(np.argmax(tramo))
        eventos.append({
            'ticker': t,
            'fecha_inicio': fechas[i0].date(),
            'fecha_pico': fechas[ipico].date(),
            'fecha_fin': fechas[i1].date(),
            'duracion_dias': i1 - i0 + 1,
            'dias_a_pico': ipico - i0,
            'menciones_pico': int(m[ipico]),
            'menciones_evento': int(tramo.sum()),
            'base_previa_mu': round(float(mu0), 2),
            'z_inicio': round(float(z0), 1),
            'censurado': cens,
        })

cat = pd.DataFrame(eventos).sort_values(['fecha_inicio', 'ticker']).reset_index(drop=True)
cat['principal'] = (cat.duracion_dias >= MIN_DUR) & (cat.menciones_evento >= MIN_MENC)
cat.to_csv(MATRIX / 'eventos' / 'eventos_atencion_v2.csv', index=False)

catp = cat[cat.principal].reset_index(drop=True)
catp.to_csv(MATRIX / 'eventos' / 'eventos_atencion_v2_principal.csv', index=False)

print(f'catalogo completo: {len(cat)} eventos | censurados: {int(cat.censurado.sum())}')
print(f'catalogo PRINCIPAL (>= {MIN_DUR} dias y >= {MIN_MENC} menciones): {len(catp)} eventos | '
      f'tickers: {catp.ticker.nunique()} | censurados: {int(catp.censurado.sum())}')
print('duracion principal: mediana', catp.duracion_dias.median(),
      '| media', round(catp.duracion_dias.mean(), 1),
      '| p90', catp.duracion_dias.quantile(.9),
      '| max', catp.duracion_dias.max())

catalogo completo: 10417 eventos | censurados: 18
catalogo PRINCIPAL (>= 3 dias y >= 300 menciones): 2791 eventos | tickers: 1012 | censurados: 6
duracion principal: mediana 11.0 | media 16.4 | p90 32.0 | max 366


In [4]:
# Celda E3 v2 - Calibracion del catalogo principal
ANCLAS = ['GME', 'AMC', 'NOK', 'BB', 'HTZ', 'NKLA', 'BBBY', 'RDDT', 'DJT', 'SMCI', 'CLOV', 'WISH']
for t in ANCLAS:
    sub = catp[catp.ticker == t]
    print(f'--- {t}: {len(sub)} eventos principales ---')
    if len(sub):
        print(sub[['fecha_inicio', 'fecha_pico', 'fecha_fin', 'duracion_dias', 'dias_a_pico',
                   'menciones_pico', 'menciones_evento', 'censurado']].to_string(index=False))
    print()

print('top 15 eventos por menciones totales:')
print(catp.nlargest(15, 'menciones_evento')[
    ['ticker', 'fecha_inicio', 'fecha_pico', 'fecha_fin', 'duracion_dias',
     'menciones_pico', 'menciones_evento', 'censurado']].to_string(index=False))

print()
print('distribucion de duraciones del catalogo principal (dias):')
print(catp.duracion_dias.describe(percentiles=[.25, .5, .75, .9, .95]).round(1).to_string())
print()
print('eventos principales por anio de inicio:')
print(catp.assign(anio=pd.to_datetime(catp.fecha_inicio).dt.year)
          .groupby('anio').size().to_string())

--- GME: 25 eventos principales ---
fecha_inicio fecha_pico  fecha_fin  duracion_dias  dias_a_pico  menciones_pico  menciones_evento  censurado
  2020-03-11 2020-04-14 2020-04-20             41           34             491              3593      False
  2020-06-08 2020-06-09 2020-06-10              3            1             788              1446      False
  2020-08-31 2020-09-22 2020-10-01             32           22             411              3870      False
  2020-11-27 2020-11-30 2020-12-09             13            3            4079             20672      False
  2021-01-13 2021-01-28 2021-02-04             23           15          141957            865463      False
  2021-05-27 2021-05-27 2021-06-10             15            0           16151            143912      False
  2021-08-24 2021-08-24 2021-08-26              3            0            5324             12957      False
  2021-10-29 2021-10-29 2021-11-11             14            0            6020             51211    

In [5]:
# Celda E4 - Diagnostico de la sombra post-evento: GME feb-mar 2021
tramo = slice('2021-02-15', '2021-03-15')
diag = pd.DataFrame({
    'menciones': gc.loc[tramo, 'GME'],
    'mu_movil': mu.loc[tramo, 'GME'].round(0),
    'sigma_piso': sd_piso.loc[tramo, 'GME'].round(0),
    'z': z.loc[tramo, 'GME'].round(2),
})
diag['mediana_30d'] = gc['GME'].rolling(30, min_periods=15).median().shift(1).loc[tramo].round(0)
print(diag.to_string())
print()
print('umbral que exigia el detector ese dia (mu + 4*sigma):')
print((mu.loc[tramo, 'GME'] + 4 * sd_piso.loc[tramo, 'GME']).round(0).to_string())

            menciones  mu_movil  sigma_piso     z  mediana_30d
fecha                                                         
2021-02-15       2800   31433.0     35249.0 -0.81      12380.0
2021-02-16       4019   31398.0     35279.0 -0.78      12380.0
2021-02-17       3864   31436.0     35247.0 -0.78      12380.0
2021-02-18       7731   31404.0     35273.0 -0.67      12380.0
2021-02-19       5447   31356.0     35305.0 -0.73      12380.0
2021-02-20       4052   31290.0     35353.0 -0.77      12380.0
2021-02-21       3249   31116.0     35478.0 -0.79      12380.0
2021-02-22       6091   30607.0     35773.0 -0.69      11723.0
2021-02-23       5608   30445.0     35876.0 -0.69      11723.0
2021-02-24      16408   30344.0     35943.0 -0.39      11723.0
2021-02-25      34838   29854.0     36033.0  0.14      11723.0
2021-02-26      21128   29545.0     35946.0 -0.23      11723.0
2021-02-27       6401   26784.0     33102.0 -0.62      11723.0
2021-02-28       5476   22266.0     25131.0 -0.67      

In [6]:
# Celda E5 - Acciones principales del catalogo (lista para X y TikTok)
N_PRINCIPALES = 50   # ajustable segun presupuesto de X/TikTok

resumen = (catp.groupby('ticker')
           .agg(n_eventos=('ticker', 'size'),
                menciones_en_eventos=('menciones_evento', 'sum'),
                pico_maximo=('menciones_pico', 'max'),
                duracion_media=('duracion_dias', 'mean'),
                duracion_max=('duracion_dias', 'max'),
                primer_evento=('fecha_inicio', 'min'),
                ultimo_evento=('fecha_fin', 'max'))
           .sort_values('menciones_en_eventos', ascending=False))
resumen['duracion_media'] = resumen['duracion_media'].round(1)

resumen.to_csv(MATRIX / 'eventos' / 'ranking_tickers_eventos.csv')
principales = resumen.head(N_PRINCIPALES)
principales.to_csv(MATRIX / 'eventos' / f'acciones_principales_top{N_PRINCIPALES}.csv')

print(f'ranking completo: {len(resumen)} tickers -> ranking_tickers_eventos.csv')
print(f'acciones principales: top {N_PRINCIPALES} -> acciones_principales_top{N_PRINCIPALES}.csv')
print()
print(principales.to_string())
print()
# cuanta señal captura el top N (para justificar el corte en la tesis)
total = resumen.menciones_en_eventos.sum()
for n in (10, 20, 30, 50, 75, 100):
    cap = resumen.menciones_en_eventos.head(n).sum() / total
    print(f'top {n:>3}: captura {cap:.1%} de las menciones en eventos')

ranking completo: 1012 tickers -> ranking_tickers_eventos.csv
acciones principales: top 50 -> acciones_principales_top50.csv

        n_eventos  menciones_en_eventos  pico_maximo  duracion_media  duracion_max primer_evento ultimo_evento
ticker                                                                                                        
GME            25               1523052       141957            16.5            41    2020-03-11    2026-05-15
AMC            14                669296       110205            10.8            22    2020-04-11    2024-05-15
TSLA           31                647915         8538            14.9            58    2020-02-03    2026-06-17
RDDT            6                444452         7193            69.0           247    2024-03-21    2026-06-04
NVDA           30                336938         6211            19.8            64    2020-02-10    2026-06-02
AAPL           24                276099         3884            14.7            82    2020-01-28 

In [7]:
# Celda E6 - Harness de sensibilidad: detector como funcion + 7 escenarios
import numpy as np

def detectar_eventos(gcm, mum, sdm, zm, Z_ON=4.0, MIN_ABS=30, S_OFF=1.0, KAPPA=0.25,
                     K_OFF=5, GAP=5, MIN_DUR=3, MIN_MENC=300):
    eventos = []
    fechas = gcm.index
    for t in gcm.columns:
        m = gcm[t].to_numpy()
        zz = zm[t].to_numpy()
        mus = mum[t].to_numpy()
        sds = sdm[t].to_numpy()

        crudos = []
        en_evento = False
        for i in range(len(m)):
            if not en_evento:
                if not np.isnan(zz[i]) and zz[i] >= Z_ON and m[i] >= MIN_ABS:
                    en_evento = True
                    i0 = i
                    mu0, s0 = mus[i], sds[i]
                    pico_corrido = m[i]
                    bajo = 0
            else:
                if m[i] > pico_corrido:
                    pico_corrido = m[i]
                if m[i] < max(mu0 + S_OFF * s0, KAPPA * pico_corrido):
                    bajo += 1
                    if bajo >= K_OFF:
                        crudos.append((i0, i - K_OFF, False))
                        en_evento = False
                else:
                    bajo = 0
        if en_evento:
            crudos.append((i0, len(m) - 1, True))

        fusion = []
        for ev in crudos:
            if fusion and ev[0] - fusion[-1][1] <= GAP:
                fusion[-1] = (fusion[-1][0], ev[1], ev[2])
            else:
                fusion.append(ev)

        for i0, i1, cens in fusion:
            tramo = m[i0:i1 + 1]
            ipico = i0 + int(np.argmax(tramo))
            eventos.append({
                'ticker': t,
                'fecha_inicio': fechas[i0].date(),
                'fecha_pico': fechas[ipico].date(),
                'fecha_fin': fechas[i1].date(),
                'duracion_dias': i1 - i0 + 1,
                'menciones_pico': int(m[ipico]),
                'menciones_evento': int(tramo.sum()),
                'censurado': cens,
            })
    catx = pd.DataFrame(eventos)
    catx['principal'] = (catx.duracion_dias >= MIN_DUR) & (catx.menciones_evento >= MIN_MENC)
    return catx

# base robusta: mediana movil e IQR (para el escenario 'base robusta')
med = gc.rolling(VENTANA, min_periods=MIN_P).median().shift(1)
q75 = gc.rolling(VENTANA, min_periods=MIN_P).quantile(0.75).shift(1)
q25 = gc.rolling(VENTANA, min_periods=MIN_P).quantile(0.25).shift(1)
sd_rob = np.maximum(np.maximum((q75 - q25) / 1.349, np.sqrt(med)), 1.0)
z_rob = (gc - med) / sd_rob
print('base robusta lista')

ESCENARIOS = {
    'base v2':        dict(),
    'KAPPA 0.10':      dict(KAPPA=0.10),
    'KAPPA 0.50':      dict(KAPPA=0.50),
    'Z 3':            dict(Z_ON=3.0),
    'Z 5':            dict(Z_ON=5.0),
    'cortes 5d/500m': dict(MIN_DUR=5, MIN_MENC=500),
}

cats = {}
for nombre, kw in ESCENARIOS.items():
    cats[nombre] = detectar_eventos(gc, mu, sd_piso, z, **kw)
    print(f'{nombre}: {int(cats[nombre].principal.sum())} eventos principales', flush=True)

cats['base robusta'] = detectar_eventos(gc, med, sd_rob, z_rob)
print(f"base robusta: {int(cats['base robusta'].principal.sum())} eventos principales")

for nombre, c in cats.items():
    c.to_csv(MATRIX / 'eventos' / f"sens_{nombre.replace(' ', '_').replace('.', '')}.csv",
             index=False)
print('escenarios guardados en Matrix/eventos/')

base robusta lista
base v2: 2791 eventos principales
ALFA 0.10: 3272 eventos principales
ALFA 0.50: 1826 eventos principales
Z 3: 3129 eventos principales
Z 5: 2490 eventos principales
cortes 5d/500m: 1723 eventos principales
base robusta: 3451 eventos principales


OSError: Cannot save file into a non-existent directory: '/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix/eventos/sens_cortes_5d'

In [ ]:
# Celda E6b - Guardado de escenarios (corrige el nombre con diagonal)
import re
for nombre, c in cats.items():
    limpio = re.sub(r'[^A-Za-z0-9]+', '_', nombre).strip('_')
    c.to_csv(MATRIX / 'eventos' / f'sens_{limpio}.csv', index=False)
    print('guardado:', f'sens_{limpio}.csv')
print('escenarios guardados en Matrix/eventos/')

In [ ]:
# Celda E7 - Comparacion de escenarios: forma del catalogo, estabilidad y ancla GME
base = cats['base v2'][cats['base v2'].principal].reset_index(drop=True)

def estabilidad(basec, altc):
    alt_por_t = dict(tuple(altc.groupby('ticker')))
    match, difs = 0, []
    for _, e in basec.iterrows():
        c = alt_por_t.get(e.ticker)
        if c is None:
            continue
        sol = c[(c.fecha_inicio <= e.fecha_fin) & (c.fecha_fin >= e.fecha_inicio)]
        if len(sol):
            match += 1
            difs.append(abs(int(sol.iloc[0].duracion_dias) - int(e.duracion_dias)))
    return match / len(basec), (float(np.median(difs)) if difs else np.nan)

def dur_gme_enero(catx):
    d = pd.Timestamp('2021-01-28').date()
    s = catx[(catx.ticker == 'GME') & (catx.fecha_inicio <= d) & (catx.fecha_fin >= d)]
    return int(s.iloc[0].duracion_dias) if len(s) else None

filas = []
for nombre, c in cats.items():
    p = c[c.principal]
    tasa, dmed = estabilidad(base, p)
    filas.append({
        'escenario': nombre,
        'eventos': len(p),
        'tickers': p.ticker.nunique(),
        'dur_mediana': p.duracion_dias.median(),
        'dur_p90': p.duracion_dias.quantile(.9),
        'dur_max': p.duracion_dias.max(),
        'censurados': int(p.censurado.sum()),
        'pct_base_recuperado': round(100 * tasa, 1),
        'mediana_dif_duracion': dmed,
        'GME_ene21_dias': dur_gme_enero(p),
    })

comp = pd.DataFrame(filas)
comp.to_csv(MATRIX / 'eventos' / 'sensibilidad_resumen.csv', index=False)
print(comp.to_string(index=False))

In [ ]:
# Celda E8 - Que eventos extra ve la base robusta que el base v2 no ve (caso GME)
rob = cats['base robusta'][cats['base robusta'].principal]
print('eventos GME en base robusta durante 2021:')
g21 = rob[(rob.ticker == 'GME') &
          (rob.fecha_inicio >= pd.Timestamp('2021-01-01').date()) &
          (rob.fecha_inicio <= pd.Timestamp('2021-12-31').date())]
print(g21[['fecha_inicio', 'fecha_pico', 'fecha_fin', 'duracion_dias',
           'menciones_pico', 'menciones_evento']].to_string(index=False))

In [ ]:
# Celda E9 - Autopsia del evento RDDT de 247 dias (2024-10-29 a 2025-07-02)
ini, fin = pd.Timestamp('2024-10-29'), pd.Timestamp('2025-07-02')
serie = gc['RDDT']

# parametros que vio el detector al arrancar
mu0 = float(mu.at[ini, 'RDDT'])
s0 = float(sd_piso.at[ini, 'RDDT'])
print(f'base pre-evento: mu0 = {mu0:.0f}, sigma0 = {s0:.0f}')
print(f'menciones el dia de encendido: {int(serie.at[ini])}')

# reconstruir el umbral de extincion dia a dia (pico corrido)
tramo = serie.loc[ini:fin]
pico_corrido = tramo.cummax()
umbral = np.maximum(mu0 + 1.0 * s0, 0.25 * pico_corrido)
bajo = tramo < umbral

# secuencias de dias consecutivos bajo el umbral (el evento muere con 5)
racha = (bajo.groupby((~bajo).cumsum()).cumcount() + 1) * bajo
print(f'\nracha maxima de dias bajo el umbral durante el evento: {int(racha.max())} (muere con 5)')
casi = racha[racha >= 3]
print('momentos de casi-muerte (3+ dias consecutivos bajo umbral):')
for f, r in casi.items():
    print(f'  {f.date()} (racha {int(r)}, menciones {int(tramo.at[f])}, umbral {umbral.at[f]:.0f})')

# forma del evento: promedio semanal de menciones
print('\npromedio diario de menciones por semana:')
sem = tramo.resample('W').mean().round(0).astype(int)
print(sem.to_string())

# el mismo periodo en el escenario kappa 0.50 (muerte facil): ¿en cuantos eventos lo parte?
k50 = cats['KAPPA 0.50'][cats['KAPPA 0.50'].principal]
r50 = k50[(k50.ticker == 'RDDT') &
          (k50.fecha_fin >= ini.date()) & (k50.fecha_inicio <= fin.date())]
print('\nel mismo periodo bajo kappa 0.50 (muerte facil):')
print(r50[['fecha_inicio', 'fecha_pico', 'fecha_fin', 'duracion_dias',
           'menciones_pico', 'menciones_evento']].to_string(index=False))

In [ ]:
# Celda E10 - Censo de eventos de baja amplitud: ¿cuantos RDDT hay en el catalogo?
catp2 = catp.copy()
catp2['amplitud'] = (catp2.menciones_pico / catp2.base_previa_mu.clip(lower=1)).round(1)
catp2['piso_gana'] = catp2.menciones_pico * 0.25 < (catp2.base_previa_mu * 2)  # aprox: pico chico vs base

print('distribucion de amplitud (pico / base pre-evento):')
print(catp2.amplitud.describe(percentiles=[.1, .25, .5, .75, .9]).round(1).to_string())

# la esquina sospechosa: larga duracion Y baja amplitud
sospechosos = catp2[(catp2.duracion_dias >= 60) & (catp2.amplitud < 8)]
print(f'\neventos largos (>= 60 dias) y de baja amplitud (< 8x): {len(sospechosos)}')
print(sospechosos.sort_values('duracion_dias', ascending=False)[
    ['ticker', 'fecha_inicio', 'fecha_fin', 'duracion_dias', 'menciones_pico',
     'base_previa_mu', 'amplitud', 'censurado']].head(25).to_string(index=False))

# contraste: los largos de ALTA amplitud (burbujas largas legitimas)
legitimos = catp2[(catp2.duracion_dias >= 60) & (catp2.amplitud >= 8)]
print(f'\neventos largos (>= 60 dias) de alta amplitud (>= 8x): {len(legitimos)}')
print(legitimos.sort_values('duracion_dias', ascending=False)[
    ['ticker', 'fecha_inicio', 'fecha_fin', 'duracion_dias', 'menciones_pico',
     'base_previa_mu', 'amplitud']].head(15).to_string(index=False))

catp2.to_csv(MATRIX / 'eventos' / 'eventos_atencion_v2_principal_amplitud.csv', index=False)
print('\ncatalogo principal con columna de amplitud guardado')

In [8]:
# Celda E11 - Flag de regimen lento y catalogo final
catp2['regimen_lento'] = (catp2.duracion_dias >= 60) & (catp2.amplitud < 8)

print(f'eventos marcados regimen_lento: {int(catp2.regimen_lento.sum())} de {len(catp2)}')
print(catp2[catp2.regimen_lento][['ticker', 'fecha_inicio', 'fecha_fin',
      'duracion_dias', 'amplitud']].to_string(index=False))

catp2.to_csv(MATRIX / 'eventos' / 'eventos_atencion_v2_principal_final.csv', index=False)
print('\ncatalogo FINAL guardado: eventos_atencion_v2_principal_final.csv')
print('columnas:', list(catp2.columns))

# resumen del catalogo de trabajo (excluyendo regimen lento, como corrida de robustez futura)
sin_lentos = catp2[~catp2.regimen_lento]
print(f'\ncatalogo sin regimen lento: {len(sin_lentos)} eventos | '
      f'mediana {sin_lentos.duracion_dias.median():.0f} dias | '
      f'p90 {sin_lentos.duracion_dias.quantile(.9):.0f} | max {sin_lentos.duracion_dias.max()}')

NameError: name 'catp2' is not defined

In [9]:
exec(open('/Users/ppizam/Claude/Master Thesis/Desarrollo/Metodologia/Matrix/celda_K1.py').read())

re-corriendo detector con campos completos...

--- 1. forma de los catalogos (solo principales) ---
kappa 0.25 (base): 2791 eventos | 1012 tickers | mediana 11 | p90 32 | max 366 | censurados 6
kappa 0.10: 3272 eventos | 1177 tickers | mediana 16 | p90 52 | max 1602 | censurados 15
kappa 0.50: 1826 eventos | 780 tickers | mediana 8 | p90 18 | max 128 | censurados 3

--- 2. ancla GME (ene-2021) ---
kappa 0.25: GME inicia 2021-01-13 00:00:00 pico 2021-01-28 fin 2021-02-04 00:00:00 (23 dias)
kappa 0.50: GME inicia 2021-01-13 pico 2021-01-28 fin 2021-02-02 (21 dias)

--- 3. cobertura del panel B(t) existente sobre los nuevos catalogos ---
kappa 0.50: 1826 eventos principales | 93.0% con inicio identico a un evento base | 100.0% con TODOS sus dias en el panel B(t) | 100.0% de los dias-evento cubiertos
kappa 0.10: 3272 eventos principales | 80.0% con inicio identico a un evento base | 57.8% con TODOS sus dias en el panel B(t) | 62.4% de los dias-evento cubiertos

lectura: kappa 0.50 deberia 